# 04 — Semantic Search

## Goal

Replace the current lexical search approach with semantic search using embeddings.

This notebook will:

- Generate embeddings for document chunks.
- Generate an embedding for the user query.
- Compare the query with the document chunks.
- Retrieve the most semantically relevant results.
- Compare semantic search with the current MinSearch implementation.

## Why Semantic Search?

The current retrieval system relies mainly on lexical matching.

This works well when the query and the documents use similar words, but it may fail when they express the same idea using different vocabulary.

Semantic search represents text as numerical vectors called embeddings. Texts with similar meanings should appear close to each other in the vector space, even when they do not contain the same exact words.

## Retrieval Architecture

```text
User Question
      ↓
Query Embedding
      ↓
Vector Similarity Search
      ↓
Relevant Document Chunks
      ↓
Prompt Construction
      ↓
Large Language Model
      ↓
Answer

## Embedding Model Selection

For the semantic retrieval system, we will use OpenAI's `text-embedding-3-small` model.

This model converts text into numerical vectors called embeddings. Texts with similar meanings should produce vectors that are close to each other in the embedding space.

### Why this model?

- It provides strong semantic retrieval quality.
- It is easy to integrate using the OpenAI Python SDK.
- It is suitable for a portfolio-scale RAG application.
- It allows the retrieval architecture to remain independent from the language model used to generate the final answer.
- The embedding provider can be replaced later without redesigning the complete RAG pipeline.

### Architecture decision

Revenue AI Copilot will use two separate model providers:

- **OpenAI** for text embeddings.
- **Groq** for answer generation.

This separation allows each model to be selected according to its specific role.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Generate Embeddings

In [2]:
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="Revenue Management is the practice of selling the right room to the right customer at the right time."
)

embedding = response.data[0].embedding

print(type(embedding))
print(len(embedding))
print(embedding[:10])

<class 'list'>
1536
[-0.016021728515625, 0.002277374267578125, 0.046173095703125, 0.00862884521484375, 0.00777435302734375, -0.004131317138671875, -0.051361083984375, 0.0101318359375, 0.006740570068359375, -0.0025615692138671875]


In [3]:
def get_embedding(text, model="text-embedding-3-small"):
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

In [4]:
test_embedding = get_embedding(
    "How can a hotel increase revenue during low-demand periods?"
)

print(type(test_embedding))
print(len(test_embedding))

<class 'list'>
1536


### Embedding Function

The `get_embedding()` function converts text into a 1536-dimensional numerical vector using OpenAI's `text-embedding-3-small` model.

Texts with similar meanings should produce vectors that are located close to each other in the embedding space.

In [5]:
text_1 = "Hotels can increase revenue by adjusting prices according to demand."
text_2 = "Dynamic pricing helps hotels optimize room rates based on market demand."
text_3 = "The hotel restaurant serves breakfast from 7:00 to 10:00."

In [6]:
embedding_1 = get_embedding(text_1)
embedding_2 = get_embedding(text_2)
embedding_3 = get_embedding(text_3)

## Cosine Similarity

In [7]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [8]:
similarity_12 = cosine_similarity(
    [embedding_1],
    [embedding_2]
)[0][0]

similarity_13 = cosine_similarity(
    [embedding_1],
    [embedding_3]
)[0][0]

print(f"Similarity 1-2: {similarity_12:.4f}")
print(f"Similarity 1-3: {similarity_13:.4f}")

Similarity 1-2: 0.7304
Similarity 1-3: 0.3200


In [9]:
from app.ingest import load_documents, create_chunks

source_documents = load_documents("data/raw")
chunks = create_chunks(source_documents)

print("Pages loaded:", len(source_documents))
print("Chunks created:", len(chunks))

print(chunks[0])

Pages loaded: 183
Chunks created: 358
{'source': 'Hotel Revenue Guide eBook_18.07.2023.pdf', 'page': 2, 'chunk_id': 0, 'text': '2. Hotel Revenue Managementwww.amadeus-hospitality.com Hotel revenue management basics A. What is hotel revenue management? B. What is the purpose of hotel revenue management? C. Key principles of hotel revenue management D. Importance of hotel revenue management E. Hotel revenue management key performance indicators (KPIs) F. Hotel revenue management origins Market segmentation by traveler type A. Capturing leisure demand B. Capturing business demand C. Capturing bleisure demand D. Capturing group business Pricing strategies for hotels A. What is hotel pricing optimization? B. Dynamic pricing strategies C. Differentiated pricing strategies Maximizing revenue opportunities: leisure, business, and group strategies A. Offer attractive add-ons B. Ensure rate parity C. Use the right distributi'}


## Generate Embeddings for Document Chunks

Each document chunk will be converted into an embedding and stored together with its metadata.

To make the process more efficient, embeddings will be generated in batches instead of sending one API request per chunk.

In [10]:
from time import sleep

EMBEDDING_MODEL = "text-embedding-3-small"
BATCH_SIZE = 50


def get_embeddings_batch(texts, model=EMBEDDING_MODEL):
    response = client.embeddings.create(
        model=model,
        input=texts
    )
    return [item.embedding for item in response.data]

In [11]:
semantic_documents = []

for start in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[start:start + BATCH_SIZE]
    batch_texts = [chunk["text"] for chunk in batch]

    batch_embeddings = get_embeddings_batch(batch_texts)

    for offset, (chunk, embedding) in enumerate(zip(batch, batch_embeddings)):
        semantic_documents.append({
            "id": start + offset,
            "source": chunk["source"],
            "page": chunk["page"],
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "embedding": embedding
        })

    print(
        f"Processed {min(start + BATCH_SIZE, len(chunks))}"
        f"/{len(chunks)} chunks"
    )

    sleep(0.2)

Processed 50/358 chunks
Processed 100/358 chunks
Processed 150/358 chunks
Processed 200/358 chunks
Processed 250/358 chunks
Processed 300/358 chunks
Processed 350/358 chunks
Processed 358/358 chunks


In [12]:
print("Semantic documents:", len(semantic_documents))
print("Embedding dimensions:", len(semantic_documents[0]["embedding"]))

print({
    key: value
    for key, value in semantic_documents[0].items()
    if key != "embedding"
})

Semantic documents: 358
Embedding dimensions: 1536
{'id': 0, 'source': 'Hotel Revenue Guide eBook_18.07.2023.pdf', 'page': 2, 'chunk_id': 0, 'text': '2. Hotel Revenue Managementwww.amadeus-hospitality.com Hotel revenue management basics A. What is hotel revenue management? B. What is the purpose of hotel revenue management? C. Key principles of hotel revenue management D. Importance of hotel revenue management E. Hotel revenue management key performance indicators (KPIs) F. Hotel revenue management origins Market segmentation by traveler type A. Capturing leisure demand B. Capturing business demand C. Capturing bleisure demand D. Capturing group business Pricing strategies for hotels A. What is hotel pricing optimization? B. Dynamic pricing strategies C. Differentiated pricing strategies Maximizing revenue opportunities: leisure, business, and group strategies A. Offer attractive add-ons B. Ensure rate parity C. Use the right distributi'}
